In [1]:
import requests, hashlib, time, os, json
from pathlib import Path
from datetime import datetime
from math import log10
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# load_dotenv()
# API_KEY    = os.getenv('PODCASTINDEX_API_KEY')
# API_SECRET = os.getenv('PODCASTINDEX_API_SECRET')
# if not API_KEY or not API_SECRET:
#     raise ValueError('Add PODCASTINDEX_API_KEY and PODCASTINDEX_API_SECRET to your .env file.')

# BASE_URL   = 'https://api.podcastindex.org/api/1.0'
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

This is the function clean description. I removed fuzzy matching and adding HTML tag removal

In [2]:
import re
import string
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
DEFAULT_STOPWORDS = nlp.Defaults.stop_words | {"podcast", "episode"}

def clean_description(text, stopwords=DEFAULT_STOPWORDS, do_lower=True):
    """
    Preprocess text by:
    - Removing HTML tags
    - Lowercasing (optional)
    - Removing punctuation
    - Lemmatization
    - Removing stopwords
    """
    if not isinstance(text, str):
        return ''

    # Drop HTML tags like <p>, </div>, etc.
    text = re.sub(r'<[^>]+>', ' ', text)

    if do_lower:
        text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and token.text not in stopwords]

    result = ' '.join(words)
    result = re.sub(r'\s+', ' ', result).strip()
    return result

In [ ]:
INPUT_PATH = OUTPUT_DIR / 'podcasts_cleaned2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'podcasts_cleanedHTML2csv'
CHUNKSIZE = 50

first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} rows...")

print(f"Saved cleaned podcast data to {OUTPUT_PATH}")


Process episodes.

In [16]:
INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
OUTPUT_PATH = OUTPUT_DIR / 'episodes_cleanedHTML2.csv'
CHUNKSIZE = 50
first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} episode rows...")

print(f"Saved cleaned episode data to {OUTPUT_PATH}")

Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 e

Other person runs this: ESTELLA run this one - NOT USING THIS PART

In [ ]:
# INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
# OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedpart1.csv'
# OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedpart2.csv'
# first = False
# for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
#     chunk['clean_description'] = chunk['description'].apply(clean_description)
#     chunk.to_csv(OUTPUT_PATH2, mode='w' if first else 'a', index=False, header=first)
#     first = False
#     print(f"Processed {len(chunk)} episode rows...")

# print(f"Saved cleaned episode data to {OUTPUT_PATH2}")

In [ ]:
#Combine back into one csv called episodes_cleaned2HTML.csv
# OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedHTML1.csv'
# OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedHTML2.csv'
# df1 = pd.read_csv(OUTPUT_PATH1)
# df2 = pd.read_csv(OUTPUT_PATH2)
# combined_df = pd.concat([df1, df2], ignore_index=True)
# combined_df.to_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv', index=False)
# print(f"Combined cleaned episode data saved to {OUTPUT_DIR / 'episodes_cleanedHTML2.csv'}")

Open the csv files and check that most tags are removed beore continuing please please please

Get embeddings now with the cleaned html csv  files. First for podcasts

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleanedHTML2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv')

# Build text strings: for podcasts, combine name + description + categories; for episodes, combine episode name + description.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('clean_description', '')}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('clean_description', '')}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Podcast SVD embeddings
N_COMPONENTS = 100  # number of SVD dimensions 100-300 for now cuz um thats what i learned

tfidf_shows = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_shows = tfidf_shows.fit_transform(show_texts) # sparse (n_podcasts, 10000)
print(f'Podcast TF-IDF matrix: {tfidf_matrix_shows.shape}')

svd_shows = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs = svd_shows.fit_transform(tfidf_matrix_shows)  # dense (n_podcasts, 100)
print(f'Explained variance: {svd_shows.explained_variance_ratio_.sum():.2%}')

np.save(OUTPUT_DIR / 'podcasts_embeddings.npy', show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'podcasts_embeddings_ids.txt').write_text('\n'.join(show_ids))
print(f'Podcast embeddings: shape={show_embs.shape} | {(OUTPUT_DIR/"podcasts_embeddings.npy").stat().st_size/1024:.0f} KB')

Podcast TF-IDF matrix: (750, 8988)
Explained variance: 36.45%
Podcast embeddings: shape=(750, 100) | 293 KB


Create episode embeddings

In [7]:
# Episode SVD embeddings:
def episode_embeddings(file_num):
    # df_episodes = pd.read_csv(OUTPUT_DIR / f'episodes_cleanedHTML{file_num}.csv')
    tfidf_eps = TfidfVectorizer(max_features=10000, stop_words='english')
    tfidf_matrix_eps = tfidf_eps.fit_transform(ep_texts)
    print(f'Episode TF-IDF matrix: {tfidf_matrix_eps.shape}')

    svd_eps = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
    ep_embs = svd_eps.fit_transform(tfidf_matrix_eps)

    np.save(OUTPUT_DIR / f'episodes_embeddings.npy', ep_embs.astype(np.float32))
    Path(OUTPUT_DIR / f'episodes_embeddings_ids.txt').write_text('\n'.join(ep_ids))
    print(f'Episode embeddings:  shape={ep_embs.shape} | {(OUTPUT_DIR/f"episodes_embeddings.npy").stat().st_size/1024:.0f} KB')

    # Save vectorizers + SVD models (needed at query time!) 
    import pickle
    # for podcasts saving
    with open(OUTPUT_DIR / f'svd_podcasts.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows}, f)
        
    # for episodes saving
    with open(OUTPUT_DIR / f'svd_episodes.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_eps, 'svd': svd_eps}, f)
    print(f'Saved svd_podcasts.pkl and svd_episodes.pkl')
    

episode_embeddings(2)

Episode TF-IDF matrix: (13930, 10000)
Episode embeddings:  shape=(13930, 100) | 5442 KB
Saved svd_podcasts.pkl and svd_episodes.pkl


Now get the mixed embeddings

In [ ]:
# Build mixed podcast embeddings in the SAME latent space as podcast SVD
import pickle
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load embeddings and IDs already created above
show_embs = np.load(OUTPUT_DIR / 'podcasts_embeddings.npy')  # (n_podcasts, n_components)
show_ids = [line.strip() for line in (OUTPUT_DIR / 'podcasts_embeddings_ids.txt').read_text().splitlines()]

# Load cleaned data
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleanedHTML2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv')

# Aggregate episode text by podcast
podcast_to_episode_text = defaultdict(list)
for row in df_episodes.itertuples(index=False):
    podcast_to_episode_text[str(row.podcast_id)].append(f"{row.episode_name} {row.clean_description}")

# Create one aggregated episode document per podcast in podcast order
episode_docs_by_show = []
for sid in show_ids:
    episode_docs_by_show.append(" ".join(podcast_to_episode_text.get(str(sid), [])))

# Project aggregated episode docs into the EXISTING podcast TF-IDF + SVD space
episode_tfidf_in_show_space = tfidf_shows.transform(episode_docs_by_show)
episode_embs_in_show_space = svd_shows.transform(episode_tfidf_in_show_space)

# L2-normalize both components before mixing
show_embs_norm = normalize(show_embs, norm='l2')
episode_embs_norm = normalize(episode_embs_in_show_space, norm='l2')

# Weighted mix in one shared space
alpha_episode = 0.1
alpha_show = 0.9
new_show_embs = alpha_episode * episode_embs_norm + alpha_show * show_embs_norm
new_show_embs = normalize(new_show_embs, norm='l2')

# Save mixed embeddings + IDs
np.save(OUTPUT_DIR / 'embeddings_mixed.npy', new_show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'podcasts_embeddings_ids.txt').write_text('\n'.join(show_ids))
print(f"Saved mixed podcast embeddings: shape={new_show_embs.shape} | {(OUTPUT_DIR/'embeddings_mixed.npy').stat().st_size/1024:.0f} KB")

# Save query-time model for mixed retrieval (same vectorizer/SVD used to embed queries)
with open(OUTPUT_DIR / 'svd_mixed.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows, 'mix_weights': {'episode': alpha_episode, 'show': alpha_show}}, f)

print('Saved svd_mixed.pkl for query-time projection in shared space.')

Saved new podcast embeddings (0.1*episode centroid + 0.9*description)
